# Data Normalization: SOURCE + TARGET

- SOURCE: Log10 + Z-score
- TARGET: Log10 only (value only, Y ignored). Z-score is applied in 07 with per-fold train statistics

In [1]:
import pandas as pd
import numpy as np
import os

source_dir = r'../data/in_vitro_data_deduplicated'
target_dir = r'../data/pksmarts_data'
output_dir = r'../data/normalized_data'
os.makedirs(output_dir, exist_ok=True)

print("Data Normalization\n")

def get_value_column(df):
    if 'value' in df.columns:
        return 'value'
    elif 'Y' in df.columns:
        return 'Y'
    return None

# Files whose values are already log10-transformed (Biogen: 'LOG HLM_CLint (mL/min/kg)')
ALREADY_LOG_FILES = {'biogen_CL_raw_smart_deduplicated.csv'}

# Per-file row filter: drop in vivo / unclassified measurements and mixed units
# ChEMBL CL → human in vitro microsomal CLint (mL.min-1.g-1 = uL/min/mg protein)
# ChEMBL Fu → human plasma/serum fraction unbound (unitless, 0 < fu <= 1)
#   Fu is an in vitro binding assay, but 01's domain-classification regex tags most of it 'unknown',
#   so filter by assay description (desc) instead of domain
FU_MATRIX = r'plasma|serum'
FU_EXCLUDE = (r'brain|tissue|homogenate|microsom|hepatocyte|blood|liver|csf|cerebrospinal|albumin'  # other matrices
              r'|administ|dosed|dosing|oral|\biv\b|intravenous|patient|volunteer|subject')        # after dosing (ex vivo)

def fu_filter(df):
    desc = df['desc'].fillna('')
    return (
        df['units'].isna()
        & (df['species'] == 'human')
        & desc.str.contains(FU_MATRIX, case=False)
        & ~desc.str.contains(FU_EXCLUDE, case=False)
        & (df['value'] > 0) & (df['value'] <= 1)
    )

# ChEMBL T1/2 → human liver microsome (HLM) half-life (unit hr)
#   (TDC Half_Life_Obach is human in vivo half-life and shares 67 molecules with the target, so it was replaced)
T12_MATRIX = r'liver microsom|hepatic microsom|HLM'
T12_EXCLUDE = (r'hepatocyte|plasma|blood|S9|intestin|recombinant|CYP\d|supersome'
               r'|GSH|glutathione|absence of NADPH|without NADPH')

def t12_filter(df):
    desc = df['desc'].fillna('')
    return (
        (df['domain'] == 'in_vitro')
        & (df['species'] == 'human')
        & (df['units'] == 'hr')
        & desc.str.contains(T12_MATRIX, case=False)
        & ~desc.str.contains(T12_EXCLUDE, case=False)
    )

ROW_FILTERS = {
    'chembl_CL_raw_smart_deduplicated.csv': dict(domain='in_vitro', species='human', units='mL.min-1.g-1'),
    'chembl_Fu_raw_smart_deduplicated.csv': fu_filter,
    'chembl_T1_2_raw_smart_deduplicated.csv': t12_filter,
}

def apply_row_filter(df, file):
    cond = ROW_FILTERS.get(file)
    if cond is None:
        return df
    if callable(cond):
        mask = cond(df)
    else:
        mask = pd.Series(True, index=df.index)
        for col, val in cond.items():
            mask &= df[col] == val
    return df[mask].reset_index(drop=True)

def normalize_value(df, value_col, already_log=False):
    """Log10 + Z-score normalisation (skip Log10 if already_log=True)"""
    df = df.copy()
    
    # Drop rows without a value (e.g. 434 Biogen rows with no HLM CLint measurement)
    df = df.dropna(subset=[value_col]).reset_index(drop=True)
    
    if not already_log:
        # Log10 transform
        negative = (df[value_col] <= 0).sum()
        if negative > 0:
            df = df[df[value_col] > 0].reset_index(drop=True)
        
        df[value_col] = np.log10(df[value_col])
    
    # Z-score
    mean = df[value_col].mean()
    std = df[value_col].std()
    df[value_col] = (df[value_col] - mean) / std if std > 0 else 0
    
    return df, mean, std

Data Normalization



## SOURCE normalisation

In [2]:
print("SOURCE Normalization\n")

source_files = [
    ('clearance', ['chembl_CL_raw_smart_deduplicated.csv', 'biogen_CL_raw_smart_deduplicated.csv']),
    ('half_life', ['chembl_T1_2_raw_smart_deduplicated.csv']),
    ('fu', ['chembl_Fu_raw_smart_deduplicated.csv']),
]

for prop, files in source_files:
    print(f"{prop}:")
    dfs = []
    
    for file in files:
        path = os.path.join(source_dir, file)
        if not os.path.exists(path):
            print(f"  ⚠️ {file} not found")
            continue
        
        df = pd.read_csv(path)
        orig_rows = len(df)
        df = apply_row_filter(df, file)
        filt_rows = len(df)
        
        val_col = get_value_column(df)
        if val_col is None:
            print(f"  ✗ {file}: no value column")
            continue
        
        df, mean, std = normalize_value(df, val_col, already_log=file in ALREADY_LOG_FILES)
        dfs.append(df)
        
        print(f"  ✓ {file}: {orig_rows} → {filt_rows} (filter) → {len(df)} rows")
        print(f"    {val_col}: mean={mean:.4f}, std={std:.4f}" + (" (already log, Log10 skipped)" if file in ALREADY_LOG_FILES else ""))
    
    if dfs:
        combined = pd.concat(dfs, ignore_index=True)
        out_path = os.path.join(output_dir, f"{prop}_source.csv")
        combined.to_csv(out_path, index=False)
        print(f"  → {len(combined)} rows saved\n")

SOURCE Normalization

clearance:
  ✓ chembl_CL_raw_smart_deduplicated.csv: 70202 → 11425 (filter) → 11300 rows
    value: mean=1.5701, std=0.9301
  ✓ biogen_CL_raw_smart_deduplicated.csv: 3521 → 3521 (filter) → 3087 rows
    value: mean=1.3200, std=0.6240 (already log, Log10 skipped)
  → 14387 rows saved

half_life:
  ✓ chembl_T1_2_raw_smart_deduplicated.csv: 79483 → 9206 (filter) → 9204 rows
    value: mean=-0.2660, std=0.6369
  → 9204 rows saved

fu:
  ✓ chembl_Fu_raw_smart_deduplicated.csv: 13354 → 3783 (filter) → 3783 rows
    value: mean=-1.3516, std=0.8640
  → 3783 rows saved



## TARGET Log10 transform (Z-score applied per fold in 07, Y ignored)

In [3]:
print("TARGET Log10 only (Z-score per fold in 07, ignoring Y)\n")

target_files = [
    ('clearance', ['clearance_target_train.csv', 'clearance_target_val.csv', 'clearance_target_test.csv']),
    ('half_life', ['half_life_target_train.csv', 'half_life_target_val.csv', 'half_life_target_test.csv']),
    ('fu', ['fu_target_train.csv', 'fu_target_val.csv', 'fu_target_test.csv']),
]

for prop, files in target_files:
    print(f"{prop}:")
    dfs = []
    
    for file in files:
        path = os.path.join(target_dir, file)
        if not os.path.exists(path):
            print(f"  ⚠️ {file} not found")
            continue
        
        df = pd.read_csv(path)
        orig_rows = len(df)
        
        # value gets Log10 only
        # Z-score is applied in 07 with per-fold train statistics (prevents test leakage)
        if 'value' in df.columns:
            df = df.dropna(subset=['value'])
            df = df[df['value'] > 0].reset_index(drop=True)
            df['value'] = np.log10(df['value'])
        
        # Drop the Y column
        if 'Y' in df.columns:
            df = df.drop('Y', axis=1)
        
        dfs.append(df)
        print(f"  ✓ {file}: {orig_rows} → {len(df)} rows")
    
    if dfs:
        combined = pd.concat(dfs, ignore_index=True)
        out_path = os.path.join(output_dir, f"{prop}_target.csv")
        combined.to_csv(out_path, index=False)
        print(f"  → {len(combined)} rows saved (Y removed)\n")

TARGET Log10 only (Z-score per fold in 07, ignoring Y)

clearance:
  ✓ clearance_target_train.csv: 616 → 616 rows
  ✓ clearance_target_val.csv: 154 → 154 rows
  ✓ clearance_target_test.csv: 257 → 257 rows
  → 1027 rows saved (Y removed)

half_life:
  ✓ half_life_target_train.csv: 698 → 698 rows
  ✓ half_life_target_val.csv: 175 → 175 rows
  ✓ half_life_target_test.csv: 291 → 291 rows
  → 1164 rows saved (Y removed)

fu:
  ✓ fu_target_train.csv: 212 → 212 rows
  ✓ fu_target_val.csv: 53 → 53 rows
  ✓ fu_target_test.csv: 88 → 88 rows
  → 353 rows saved (Y removed)



## Validation

In [4]:
print("\n" + "="*100)
print("Verification")
print("="*100)

for prop in ['clearance', 'half_life', 'fu']:
    src_path = os.path.join(output_dir, f"{prop}_source.csv")
    tgt_path = os.path.join(output_dir, f"{prop}_target.csv")
    
    print(f"\n{prop.upper()}:")
    
    if os.path.exists(src_path):
        src = pd.read_csv(src_path)
        val_col = get_value_column(src)
        print(f"  SOURCE: {len(src)} rows")
        if val_col:
            print(f"    {val_col}: mean={src[val_col].mean():.6f}, std={src[val_col].std():.6f}")
    
    if os.path.exists(tgt_path):
        tgt = pd.read_csv(tgt_path)
        print(f"  TARGET: {len(tgt)} rows")
        print(f"    Columns: {list(tgt.columns)}")
        if 'value' in tgt.columns:
            print(f"    value: mean={tgt['value'].mean():.6f}, std={tgt['value'].std():.6f}")
        if 'Y' in tgt.columns:
            print(f"    ⚠️ Y column still present!")


Verification

CLEARANCE:
  SOURCE: 14387 rows
    value: mean=0.000000, std=0.999965
  TARGET: 1027 rows
    Columns: ['smiles', 'value', 'ik']
    value: mean=0.606081, std=0.650412

HALF_LIFE:
  SOURCE: 9204 rows
    value: mean=0.000000, std=1.000000
  TARGET: 1164 rows
    Columns: ['smiles', 'value', 'ik']
    value: mean=0.660949, std=0.622091

FU:
  SOURCE: 3783 rows
    value: mean=-0.000000, std=1.000000
  TARGET: 353 rows
    Columns: ['smiles', 'value', 'ik']
    value: mean=-1.005022, std=0.805461
